# S6E11 · Credit-card fraud: how I got to 0.9509 CV AUC

This notebook is the whole path, in the order I took it:
- what I looked at,
- what I tried,
- what I changed because of each result.

Every number printed below is produced by this notebook, with one exception: the **experiment log** in section 7 lists runs from my local machine (22-core CPU + RTX 3050), and is labelled as such.

* **Metric:** ROC-AUC on predicted probabilities, so only the *ranking* of rows matters.
* **Validation:** `StratifiedKFold(5, shuffle=True, random_state=42)` everywhere. Early stopping uses a 10% inner holdout of the training fold, never the scored fold.
* **Accelerator:** GPU (T4 or P100). XGBoost and CatBoost use it. Runtime is about 1–1.5 h.

**Credits.** I found the key ideas (per-value target encoding, digit keys, tiny trees, boosting from a GLM) in the competition discussions, then re-implemented and measured them myself:
- [score ladder by Yash Teotia](https://www.kaggle.com/competitions/playground-series-s6e11/discussion/755149), which credits goodpjw2008 (digit keys, tiny trees), SIDHAARTH SHREE (per-value encoding) and megayak (boosting from a GLM);
- [generator fingerprints by DavutKursun](https://www.kaggle.com/competitions/playground-series-s6e11/discussion/759313).

In [ ]:
import os, glob, time, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier, Pool
warnings.filterwarnings("ignore")

SMOKE = os.environ.get("SMOKE") == "1"          # tiny run for testing the code path
DATA = next(d for d in [os.environ.get("DATA", "")] + glob.glob("/kaggle/input/*s6e11*") + glob.glob("/kaggle/input/*")
            if d and os.path.exists(os.path.join(d, "train.csv")))
try:
    import torch; GPU = torch.cuda.is_available()
except ImportError:
    GPU = False
print("data:", DATA, "| GPU:", GPU, "| smoke:", SMOKE)

tr = pd.read_csv(f"{DATA}/train.csv"); te = pd.read_csv(f"{DATA}/test.csv")
if SMOKE:
    tr = tr.sample(40000, random_state=0).reset_index(drop=True); te = te.iloc[:5000].reset_index(drop=True)
y = tr["is_fraud"].values.astype(int); n = len(tr)
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(n), y))
LOG = []   # (step, model, oof auc)

def log(step, name, oof):
    auc = roc_auc_score(y, oof); LOG.append((step, name, round(auc, 5)))
    print(f"{name:40s} OOF AUC {auc:.5f}"); return auc

## 1 · First look at the data

In [ ]:
print(tr.shape, te.shape, "| missing:", tr.isna().sum().sum(), te.isna().sum().sum())
feats = [c for c in tr.columns if c not in ("id", "is_fraud")]
print("duplicate feature rows (train+test):", pd.concat([tr[feats], te[feats]]).duplicated().sum())
print("fraud rate:", y.mean().round(5))
NUM = ["amount_usd","hours_since_last_txn","txn_count_last_24h","distance_from_home_km","card_age_months",
       "customer_age","account_balance_usd","cvv_retry_count","velocity_score","time_of_day_hour",
       "day_of_week","merchant_risk_score","prior_disputes"]
CAT = ["merchant_category","card_type","auth_method","channel","device_type"]
BOOL = ["is_foreign_transaction","is_new_merchant","used_vpn","ip_country_mismatch",
        "billing_shipping_mismatch","is_ai_generated_scam_attempt"]
uni = pd.DataFrame({"auc": [max(a, 1-a) for a in (roc_auc_score(y, tr[c]) for c in NUM + ["id"])],
                    "n_unique": [tr[c].nunique() for c in NUM + ["id"]],
                    "train_mean": [tr[c].mean() for c in NUM + ["id"]],
                    "test_mean": [te[c].mean() for c in NUM + ["id"]]}, index=NUM + ["id"])
uni.sort_values("auc", ascending=False).round(4)

In [ ]:
rates = {c: tr.groupby(c)["is_fraud"].mean().sort_values() for c in CAT + BOOL}
fig, ax = plt.subplots(2, 6, figsize=(24, 7))
for a, (c, r) in zip(ax.ravel(), rates.items()):
    r.plot.barh(ax=a, color="tab:red"); a.set_title(c, fontsize=9); a.axvline(y.mean(), color="k", lw=.8)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
for a, c in zip(ax, ["merchant_risk_score", "velocity_score", "cvv_retry_count", "amount_usd"]):
    b = pd.qcut(tr[c].rank(method="first"), 20, labels=False)
    tr.groupby(b)["is_fraud"].mean().plot(ax=a, marker="o"); a.set_title(f"fraud rate by {c} (ventiles)")
plt.tight_layout(); plt.show()

**What I took from this.**
- The data is clean: no missing values, no duplicates, and train and test look identical.
- `id` has AUC ≈ 0.5 and a flat fraud rate over id deciles, so I **drop it**.
- The strongest signals are:
  - `merchant_risk_score`, `velocity_score` and `cvv_retry_count`;
  - auth method ("No Authentication" ≈ 6% fraud vs Biometric ≈ 0.5%);
  - the mismatch flags (IP-country ≈ 16%);
  - merchant category (Gift Cards / Crypto ≈ 7%).
- The curves are smooth and roughly monotone. That hints the labels may follow a simple, almost additive formula, which I test next.

## 2 · Baselines on raw columns

In [ ]:
Xraw = tr[NUM].copy(); Xraw_te = te[NUM].copy()
for c in BOOL: Xraw[c] = tr[c].astype(int); Xraw_te[c] = te[c].astype(int)
for c in CAT:
    cats = sorted(tr[c].unique())
    Xraw[c] = pd.Categorical(tr[c], cats); Xraw_te[c] = pd.Categorical(te[c], cats)

def cv_lgb(params, X, Xt, rounds=5000):
    oof = np.zeros(n); tp = np.zeros(len(Xt)); its = []
    for a, b in FOLDS:
        ia, ib = train_test_split(a, test_size=0.1, stratify=y[a], random_state=42)
        p = dict(objective="binary", metric="auc", verbose=-1, seed=42, **params)
        m = lgb.train(p, lgb.Dataset(X.iloc[ia], y[ia]), 300 if SMOKE else rounds,
                      valid_sets=[lgb.Dataset(X.iloc[ib], y[ib])], callbacks=[lgb.early_stopping(200, verbose=False)])
        oof[b] = m.predict(X.iloc[b], num_iteration=m.best_iteration)
        tp += m.predict(Xt, num_iteration=m.best_iteration) / 5; its.append(m.best_iteration)
    return oof, tp, its

lr = make_pipeline(ColumnTransformer([("n", StandardScaler(), NUM + BOOL), ("c", OneHotEncoder(), CAT)]),
                   LogisticRegression(max_iter=2000))
oof = np.zeros(n)
for a, b in FOLDS: oof[b] = lr.fit(tr.iloc[a], y[a]).predict_proba(tr.iloc[b])[:, 1]
log("2", "LogReg, raw columns", oof)

base = dict(learning_rate=0.05, subsample=0.8, subsample_freq=1, colsample_bytree=0.7, num_threads=os.cpu_count())
oof31, _, its = cv_lgb({**base, "num_leaves": 31, "min_child_samples": 100}, Xraw, Xraw_te)
log("2", "LightGBM 31 leaves", oof31); print("  best iterations:", its)

LightGBM beats logistic regression, but it stops after only a few hundred rounds. That means it overfits quickly.

If the labels really are close to a logistic formula, **smaller trees** should generalise better. The next cell tests that directly.

In [ ]:
for nl, mcs in [(4, 200), (8, 500), (15, 1000)]:
    o, t, its = cv_lgb({**base, "num_leaves": nl, "min_child_samples": mcs}, Xraw, Xraw_te)
    log("2", f"LightGBM {nl} leaves", o)
    if nl == 4: oof4, test4 = o, t

**Change:** smaller is better, and 4 leaves wins. In my local runs, hand-made features (log amount, amount/balance, cyclical hour, risk × velocity, flag counts, crossed categoricals) *lowered* AUC (0.94563 vs 0.94668), so I dropped them.

The signal is mostly additive. The remaining gains have to come from *how each value is represented*, not from more tree depth.

## 3 · Generator fingerprints: values and digits carry signal

The data is synthetic. Two things stand out:
- individual exact values repeat far more often than a continuous variable should;
- the decimal digits are not independent of the label.

A tree on the raw number cannot see "this exact value" or "this tenth digit". A per-value encoding can.

In [ ]:
for c in ["account_balance_usd", "hours_since_last_txn", "velocity_score"]:
    tenth = (np.round(tr[c] * 100).astype(np.int64) // 10) % 10
    r = tr.groupby(tenth)["is_fraud"].mean()
    print(f"{c:22s} unique values {tr[c].nunique():6d} / {n} rows | fraud rate by first decimal: "
          f"min {r.min():.4f} max {r.max():.4f} (x{r.max()/r.min():.2f})")

## 4 · Per-value target encoding + digit keys, strictly in-fold

**Keys:**
- every raw value, as a category;
- for each continuous column: integer part, integer/10, tenths, cents, last two decimals, leading digit and digit count.

Each key is mapped to a smoothed log-odds (m = 20).

**Leak-safe construction:**
- training rows get encodings from an *inner* 5-fold split of the training fold;
- validation and test rows get encodings fitted on the whole training fold.

In [ ]:
CONT = ["amount_usd","hours_since_last_txn","distance_from_home_km","account_balance_usd","velocity_score","merchant_risk_score"]
TWO_DEC = CONT[:4]
full = pd.concat([tr.drop(columns="is_fraud"), te], ignore_index=True)
k = {c: full[c].astype(str) for c in NUM + CAT + BOOL}
for c in CONT:
    v = np.round(full[c].values * 100).astype(np.int64); ip = v // 100
    k[f"{c}__int"] = ip; k[f"{c}__int10"] = ip // 10; k[f"{c}__tenth"] = (v // 10) % 10
    k[f"{c}__lead"] = pd.Series(ip).astype(str).str[0].values; k[f"{c}__ndig"] = pd.Series(ip).astype(str).str.len().values
    if c in TWO_DEC: k[f"{c}__cent"] = v % 10; k[f"{c}__frac"] = v % 100
KEYS = list(k); CODES = np.column_stack([pd.factorize(pd.Series(k[c]))[0] for c in KEYS]).astype(np.int32)
raw = full[NUM].copy()
for c in BOOL: raw[c] = full[c].astype(int)
for c in CAT: raw[c] = pd.factorize(full[c], sort=True)[0]
for c in TWO_DEC: raw[f"{c}_cents"] = np.round(full[c].values * 100).astype(np.int64) % 100
for c in CONT: raw[f"cnt_{c}"] = full[c].map(full[c].value_counts()).values
RAW = raw.values.astype(np.float32); R, Rt = RAW[:n], RAW[n:]; Ctr_all, Cte = CODES[:n], CODES[n:]
print(len(KEYS), "keys")

def enc(cf, yf, ca, prior, m=20):
    out = np.empty(ca.shape, np.float32)
    for j in range(cf.shape[1]):
        nk = max(cf[:, j].max(), ca[:, j].max()) + 1
        s = np.bincount(cf[:, j], weights=yf, minlength=nk); c = np.bincount(cf[:, j], minlength=nk)
        p = (s + m * prior) / (c + m); out[:, j] = np.log(p / (1 - p))[ca[:, j]]
    return out

def fold_encode(a, b, seed=0, labels=None):
    yy = y if labels is None else labels
    Ca, ya = Ctr_all[a], yy[a]; Ttr = np.empty(Ca.shape, np.float32)
    for i, j in StratifiedKFold(5, shuffle=True, random_state=seed).split(Ca, ya):
        Ttr[j] = enc(Ca[i], ya[i], Ca[j], ya[i].mean())
    return Ttr, enc(Ca, ya, Ctr_all[b], ya.mean()), enc(Ca, ya, Cte, ya.mean())

def glm(Ttr, ytr, Ts):
    sc = StandardScaler().fit(Ttr)
    m = LogisticRegression(C=1.0, solver="newton-cholesky", max_iter=100).fit(sc.transform(Ttr), ytr)
    return [m.decision_function(sc.transform(T)) for T in [Ttr] + Ts]

### Leakage check before trusting anything

If the pipeline leaks, shuffled labels will still score well above 0.5. As a contrast, the cell also runs a version that is wrong on purpose: it encodes on all rows before splitting.

In [ ]:
a, b = FOLDS[0]
ys = np.random.default_rng(0).permutation(y)
Ttr, Tva, _ = fold_encode(a, b, labels=ys)
print("label-shuffle AUC (should be ~0.5):", round(roc_auc_score(ys[b], glm(Ttr, ys[a], [Tva])[1]), 4))
Ttr, Tva, _ = fold_encode(a, b)
print("proper in-fold, fold 0          :", round(roc_auc_score(y[b], glm(Ttr, y[a], [Tva])[1]), 4))
Tall = enc(Ctr_all, y.astype(float), Ctr_all, y.mean())
print("LEAKY encode-before-split, fold 0:", round(roc_auc_score(y[b], glm(Tall[a], y[a], [Tall[b]])[1]), 4))

The shuffled labels score at chance, and the leaky version inflates the score a lot. So the in-fold version is the honest one, and everything below uses it.

## 5 · Models on the encodings

1. **GLM**: logistic regression on the standardised encoding logits.
2. **XGBoost boosted from the GLM**: depth-3 trees start from the GLM's logit (`base_margin`) and only fit what the GLM misses. I took the parameters from a 40-trial Optuna search on folds 0–1, run locally: it moved those folds from 0.95007 to 0.95035.
3. **CatBoost boosted from the GLM**: the same idea, a different tree learner.

I repeat the encodings with 3 inner-split seeds (*seed bagging*). That makes the members slightly different.

In [ ]:
XP = dict(objective="binary:logistic", eval_metric="auc", tree_method="hist", device="cuda" if GPU else "cpu",
          learning_rate=0.02, max_depth=3, min_child_weight=4.235, subsample=0.745, colsample_bytree=0.875,
          reg_lambda=2.506, reg_alpha=0.0116, gamma=0.0123, max_bin=128)
TE_SEEDS = [0] if SMOKE else [0, 1, 2]
P = {}   # name -> (oof logit, test logit)
def put(name, b, pv, pt):
    if name not in P: P[name] = [np.zeros(n), np.zeros(len(te))]
    P[name][0][b] = pv; P[name][1] += pt / 5

t0 = time.time()
for s in TE_SEEDS:
    for f, (a, b) in enumerate(FOLDS):
        Ttr, Tva, Tte = fold_encode(a, b, seed=s)
        g = glm(Ttr, y[a], [Tva, Tte])
        if s == 0: put("GLM on encodings", b, g[1], g[2])
        Xa, Xb, Xt = np.hstack([R[a], Ttr]), np.hstack([R[b], Tva]), np.hstack([Rt, Tte])
        ia, ib = train_test_split(np.arange(len(a)), test_size=0.1, stratify=y[a], random_state=42)
        d1 = xgb.DMatrix(Xa[ia], y[a][ia], base_margin=g[0][ia]); d2 = xgb.DMatrix(Xa[ib], y[a][ib], base_margin=g[0][ib])
        m = xgb.train({**XP, "seed": 42 + s}, d1, 300 if SMOKE else 20000, evals=[(d2, "v")],
                      early_stopping_rounds=300, verbose_eval=False)
        m = xgb.train({**XP, "seed": 42 + s}, xgb.DMatrix(Xa, y[a], base_margin=g[0]), int((m.best_iteration + 1) * 1.1))
        put(f"XGB from GLM (enc seed {s})", b, m.predict(xgb.DMatrix(Xb, base_margin=g[1]), output_margin=True),
            m.predict(xgb.DMatrix(Xt, base_margin=g[2]), output_margin=True))
        if s == 0:
            cp = dict(learning_rate=0.05, depth=4, l2_leaf_reg=10, eval_metric="AUC", random_seed=42, verbose=0,
                      task_type="GPU" if GPU else "CPU")
            cm = CatBoostClassifier(iterations=300 if SMOKE else 20000, od_type="Iter", od_wait=300, **cp)
            cm.fit(Pool(Xa[ia], y[a][ia], baseline=g[0][ia]), eval_set=Pool(Xa[ib], y[a][ib], baseline=g[0][ib]))
            cm = CatBoostClassifier(iterations=int((cm.get_best_iteration() + 1) * 1.1), **cp).fit(Pool(Xa, y[a], baseline=g[0]))
            put("CatBoost from GLM", b, cm.predict(Pool(Xb, baseline=g[1]), prediction_type="RawFormulaVal"),
                cm.predict(Pool(Xt, baseline=g[2]), prediction_type="RawFormulaVal"))
        print(f"enc seed {s} fold {f} done ({time.time()-t0:.0f}s)")
P["LightGBM 4 leaves (raw)"] = [oof4, test4]
for name, (o, _) in P.items(): log("5", name, o)

## 6 · Stacking

Each member's OOF prediction is turned into the logit of its rank. A strongly regularised logistic regression then learns the weights.

The stack is scored with **nested CV**: the weights are fitted on 4/5 of the OOF rows (using a different split) and scored on the remaining 1/5. So the weights never see the rows they are scored on.

In [ ]:
from scipy.special import logit
from scipy.stats import rankdata, spearmanr

def lr_rank(o, t):
    srt = np.sort(o); N = len(o)
    rt = (np.searchsorted(srt, t, "left") + np.searchsorted(srt, t, "right")) / 2
    return logit(rankdata(o) / (N + 1)), logit(np.clip(rt + .5, .5, N + .5) / (N + 1))

names = list(P); Z = [lr_rank(*P[k]) for k in names]
O = np.column_stack([z[0] for z in Z]); T = np.column_stack([z[1] for z in Z])
print("rank correlation between members:")
print(pd.DataFrame(O, columns=names).corr("spearman").round(3).to_string())
stk = lambda: LogisticRegression(C=0.1, solver="newton-cholesky", max_iter=200)
nested = np.zeros(n)
for a, b in StratifiedKFold(5, shuffle=True, random_state=7).split(O, y):
    nested[b] = stk().fit(O[a], y[a]).decision_function(O[b])
log("6", "STACK (nested CV)", nested)
final = stk().fit(O, y)
print(dict(zip(names, final.coef_[0].round(3))))

In [ ]:
sub = pd.DataFrame({"id": te["id"], "is_fraud": 1 / (1 + np.exp(-final.decision_function(T)))})
assert list(sub.columns) == ["id", "is_fraud"] and sub.is_fraud.between(0, 1).all()
assert not sub.isna().any().any() and not sub.id.duplicated().any()
if not SMOKE: assert len(sub) == 299_984
sub.to_csv("submission.csv", index=False)
print(sub.shape, "mean prediction", sub.is_fraud.mean().round(5), "(train fraud rate", y.mean().round(5), ")")
pd.DataFrame(LOG, columns=["section", "model", "OOF ROC-AUC"])

## 7 · Everything else I tried (local runs, same folds)

My full local run trained 22 models; their stack scored **0.95091** nested CV, which is my best submission (sub02). Things that did **not** help are kept here on purpose:

| Experiment | OOF AUC | Verdict |
|---|---:|---|
| LightGBM 31 leaves + hand-made domain features | 0.94563 | worse than raw (0.94668): dropped |
| LightGBM 4 leaves with `is_unbalance` (class weighting) | 0.94674 | worse than unweighted (0.94759) |
| Logistic regression, `class_weight="balanced"` | 0.94190 | same as unweighted (0.94187) |
| GAM-style logistic regression (50 quantile bins / feature) | 0.94459 | additive part alone is not enough |
| Jointly fitted one-hot GLM on all keys (fold 0) | 0.9473 | worse than GLM on encodings (0.9481) |
| Pair keys (category × risk etc.) added to XGB-GLM | 0.95057 | +0.00003, negligible |
| Encoding smoothing m = 80 instead of 20 | 0.95063 | no change |
| LightGBM 3 leaves boosted from GLM | 0.95064 | best single model |
| CatBoost on encodings (no GLM start) | 0.94965 | useful only as a stack member |
| RealMLP (pytabkit), raw + digit categories | 0.94428 | weak, ~0 stack weight |
| RandomForest / ExtraTrees | 0.93277 / 0.93620 | weak |
| HistGradientBoosting / XGBoost / CatBoost on raw | 0.94645 / 0.94744 / 0.94633 | below 4-leaf LightGBM |

One bug worth recording: my first "LightGBM from GLM" scored 0.5985, because LightGBM does not store `init_score` in the model. The GLM logit has to be added back at predict time. After the fix it scored 0.95064.

## 8 · Leaderboard and conclusions

* Public LB of my submissions: **Rank 1005 as of 22/11/2026**
* The public LB uses about 20% of the test set (≈1,000 frauds). A community resampling study found one submission moves ±0.003 by chance, so I choose finals by CV.
* What worked, in order of gain: tiny trees → in-fold per-value + digit-key encodings → boosting from a GLM → seed bagging → stacking.
* Next ideas: a GPT-2 model reading rows as text, a better-tuned neural net on the encodings, and more seeds. Scripts for running those on Kaggle's GPU are in my repo.